In [1]:
import sys
from pathlib import Path
from datetime import datetime, timedelta
from dotenv import load_dotenv
import time


from tradingbot.core.sequence import sequence_builder
from tradingbot.core.candles import candle_builder
from tradingbot.core.constants import Interval
from tradingbot.indicators import (
    RelativeStrengthIndex,
    StochasticOscillator,
    StochasticRSI,
)
from tradingbot.kite import (
    KiteSessionManager,
)
from tradingbot.core.ticker import Ticker


# Resolve paths so this notebook works whether the kernel starts in the repo root or notebooks/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_HISTORIC_DATA_DIR = PROJECT_ROOT / "data" / "historic_data"
INDICATOR_DATA_DIR = PROJECT_ROOT / "data" / "historic_data_with_indicators"
RAW_HISTORIC_DATA_DIR.mkdir(parents=True, exist_ok=True)
INDICATOR_DATA_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv()
session_manager = KiteSessionManager()
session = session_manager.start_session(cli=False)



🔍 Checking for cached access token...
📖 Loaded cached access token from disk
✓ Access token validation successful
✅ Cached access token is valid!



In [2]:
STOCKS = [
    "RELIANCE",
    "HDFCBANK",
    "ICICIBANK",
    "INFY",
    "TCS",
    "SBIN",
    "AXISBANK",
    "KOTAKBANK",
    "LT",
    "ITC",
    "HINDUNILVR",
    "BHARTIARTL",
    "MARUTI",
    "M&M",
    "SUNPHARMA",
    "HCLTECH",
    "TECHM",
    "WIPRO",
    "TITAN",
    "ASIANPAINT",
    "BAJFINANCE",
    "BAJAJFINSV",
    "ULTRACEMCO",
    "NTPC",
    "POWERGRID",
    "ONGC",
    "COALINDIA",
    "TATASTEEL",
    "JSWSTEEL",
    "HINDALCO",
    "GRASIM",
    "ADANIPORTS",
    "DRREDDY",
    "CIPLA",
    "EICHERMOT",
    "HEROMOTOCO",
    "BAJAJ-AUTO",
    "NESTLEIND",
]

In [3]:
# Download historic data with resume/skip support
import csv
from tradingbot.core.ticker import HISTORIC_INCEPTION_START

INTERVALS = ["1m", "5m", "15m", "1h", "day"]
SLEEP_SECONDS = 1
HISTORIC_START_LABEL = HISTORIC_INCEPTION_START.isoformat().replace(":", "-").replace("+", "plus")


def historic_csv_path(symbol, interval):
    normalized_interval = Interval.normalize(interval)
    return RAW_HISTORIC_DATA_DIR / f"{symbol.upper()}_{normalized_interval}_{HISTORIC_START_LABEL}_now.csv"


def is_complete_historic_csv(path):
    if not path.exists() or not path.is_file() or path.stat().st_size == 0:
        return False

    with path.open("r", newline="") as file_obj:
        reader = csv.DictReader(file_obj)
        required_fieldnames = Ticker.historic_candle_fieldnames()
        if not reader.fieldnames or any(field not in reader.fieldnames for field in required_fieldnames):
            return False
        return next(reader, None) is not None

if "ticker" in globals() and ticker is not None:
    try:
        ticker.websocket_client.close()
    except Exception:
        pass

ticker = None

downloaded_paths = {}
skipped_paths = {}
failed_downloads = {}

for index, symbol in enumerate(STOCKS, start=1):
    downloaded_paths[symbol] = {}
    skipped_paths[symbol] = {}
    failed_downloads[symbol] = {}

    for interval in INTERVALS:
        existing_path = historic_csv_path(symbol, interval)

        if is_complete_historic_csv(existing_path):
            skipped_paths[symbol][interval] = existing_path
            print(
                f"[{index}/{len(STOCKS)}] Skipping {symbol} "
                f"{interval}; already exists -> {existing_path}"
            )
            continue

        print(
            f"[{index}/{len(STOCKS)}] Fetching {symbol} "
            f"{interval} candles since inception..."
        )
        try:
            if ticker is None:
                ticker = Ticker(symbol=symbol, timeframe=interval, session=session)

            path = ticker.get_historic_data(
                ticker_name=symbol,
                timeframe=interval,
                csv_path=existing_path,
            )
            downloaded_paths[symbol][interval] = path
            print(f"    saved -> {path}")
        except Exception as exc:
            failed_downloads[symbol][interval] = exc
            print(f"    failed -> {exc}")

        time.sleep(SLEEP_SECONDS)

    if not downloaded_paths[symbol]:
        del downloaded_paths[symbol]

    if not skipped_paths[symbol]:
        del skipped_paths[symbol]

    if not failed_downloads[symbol]:
        del failed_downloads[symbol]

if ticker is not None:
    try:
        ticker.websocket_client.close()
    except Exception:
        pass

downloaded_paths, skipped_paths, failed_downloads


[1/38] Skipping RELIANCE 1m; already exists -> /Users/akash/PycharmProjects/AlgoTrade/data/historic_data/RELIANCE_minute_1990-01-01T00-00-00plus05-30_now.csv
[1/38] Skipping RELIANCE 5m; already exists -> /Users/akash/PycharmProjects/AlgoTrade/data/historic_data/RELIANCE_5minute_1990-01-01T00-00-00plus05-30_now.csv
[1/38] Skipping RELIANCE 15m; already exists -> /Users/akash/PycharmProjects/AlgoTrade/data/historic_data/RELIANCE_15minute_1990-01-01T00-00-00plus05-30_now.csv
[1/38] Skipping RELIANCE 1h; already exists -> /Users/akash/PycharmProjects/AlgoTrade/data/historic_data/RELIANCE_60minute_1990-01-01T00-00-00plus05-30_now.csv
[1/38] Skipping RELIANCE day; already exists -> /Users/akash/PycharmProjects/AlgoTrade/data/historic_data/RELIANCE_day_1990-01-01T00-00-00plus05-30_now.csv
[2/38] Skipping HDFCBANK 1m; already exists -> /Users/akash/PycharmProjects/AlgoTrade/data/historic_data/HDFCBANK_minute_1990-01-01T00-00-00plus05-30_now.csv
[2/38] Skipping HDFCBANK 5m; already exists -> /

KeyboardInterrupt: 

In [ ]:
# Compute normalized RSI, StochRSI, and stochastic oscillator for every historical CSV with Dask
import csv
from pathlib import Path
import importlib
import dask.dataframe as dd
import pandas as pd
import tradingbot.core.indicators as core_indicators

# Reload so a long-running notebook kernel picks up local indicator changes.
core_indicators = importlib.reload(core_indicators)
CandleView = core_indicators.CandleView
RelativeStrengthIndex = core_indicators.RelativeStrengthIndex
StochasticOscillator = core_indicators.StochasticOscillator
StochasticRSI = core_indicators.StochasticRSI

HISTORIC_DATA_DIRS = [RAW_HISTORIC_DATA_DIR]
OUTPUT_SUFFIX = "_with_normalized_indicators"
INDICATOR_DATA_DIR.mkdir(parents=True, exist_ok=True)

INDICATOR_PERIODS = [7, 14]
LOOKBACK = max(period * 2 for period in INDICATOR_PERIODS)
BLOCKSIZE = "512 KiB"

OHLCV_COLUMNS = ["timestamp", "open", "high", "low", "close", "volume"]
INDICATOR_COLUMNS = [
    column
    for period in INDICATOR_PERIODS
    for column in (
        f"rsi_{period}_norm",
        f"stoch_rsi_{period}_{period}_norm",
        f"stoch_{period}_norm",
    )
]


def is_indicator_output(path):
    return path.stem.endswith(OUTPUT_SUFFIX)


def indicator_output_path(path):
    return INDICATOR_DATA_DIR / f"{path.stem}{OUTPUT_SUFFIX}.csv"


def has_indicator_columns(path):
    if not path.exists() or path.stat().st_size == 0:
        return False
    with path.open("r", newline="") as file_obj:
        reader = csv.DictReader(file_obj)
        return bool(reader.fieldnames) and all(
            column in reader.fieldnames for column in INDICATOR_COLUMNS
        )


def discover_historic_csvs():
    seen = set()
    csv_paths = []
    for directory in HISTORIC_DATA_DIRS:
        if not directory.exists():
            continue
        for path in sorted(directory.glob("*.csv")):
            if is_indicator_output(path):
                continue
            resolved = path.resolve()
            if resolved in seen:
                continue
            seen.add(resolved)
            csv_paths.append(path)
    return csv_paths


def compute_indicator_chunk(pdf):
    pdf = pdf.copy()
    if pdf.empty:
        for column in INDICATOR_COLUMNS:
            pdf[column] = pd.Series(dtype="float64")
        return pdf

    raw_candles = [
        CandleView(
            timestamp=row["timestamp"],
            open=float(row["open"]),
            high=float(row["high"]),
            low=float(row["low"]),
            close=float(row["close"]),
            volume=float(row["volume"]),
        )
        for row in pdf[OHLCV_COLUMNS].to_dict("records")
    ]

    indicators_by_period = {
        period: (
            RelativeStrengthIndex(period=period, normalize=True),
            StochasticRSI(rsi_period=period, stoch_period=period, normalize=True),
            StochasticOscillator(period=period, normalize=True),
        )
        for period in INDICATOR_PERIODS
    }
    values_by_column = {column: [] for column in INDICATOR_COLUMNS}

    for row_number in range(len(raw_candles)):
        start = max(0, row_number - LOOKBACK + 1)
        candle_window = raw_candles[start : row_number + 1]

        for period, (rsi, stoch_rsi, stoch) in indicators_by_period.items():
            values_by_column[f"rsi_{period}_norm"].append(
                rsi.compute_point(candle_window).value
            )
            values_by_column[f"stoch_rsi_{period}_{period}_norm"].append(
                stoch_rsi.compute_point(candle_window).value
            )
            values_by_column[f"stoch_{period}_norm"].append(
                stoch.compute_point(candle_window).value
            )

    for column, values in values_by_column.items():
        pdf[column] = values
    return pdf


def build_indicator_ddf(csv_path):
    ddf = dd.read_csv(csv_path, blocksize=BLOCKSIZE, assume_missing=True)
    meta = ddf._meta.copy()
    for column in INDICATOR_COLUMNS:
        meta[column] = pd.Series(dtype="float64")

    return ddf.map_overlap(
        compute_indicator_chunk,
        before=LOOKBACK - 1,
        after=0,
        meta=meta,
    )


source_csvs = discover_historic_csvs()
processed_indicator_paths = {}
skipped_indicator_paths = {}
failed_indicator_paths = {}

print(f"Found {len(source_csvs)} source CSV files")

for index, csv_path in enumerate(source_csvs, start=1):
    output_path = indicator_output_path(csv_path)

    if has_indicator_columns(output_path):
        skipped_indicator_paths[str(csv_path)] = output_path
        print(f"[{index}/{len(source_csvs)}] Skipping {csv_path}; output exists -> {output_path}")
        continue

    print(f"[{index}/{len(source_csvs)}] Computing indicators for {csv_path}")
    try:
        indicator_ddf = build_indicator_ddf(csv_path)
        indicator_ddf.to_csv(str(output_path), single_file=True, index=False)
        processed_indicator_paths[str(csv_path)] = output_path
        print(f"    saved -> {output_path}")
    except Exception as exc:
        failed_indicator_paths[str(csv_path)] = exc
        print(f"    failed -> {exc}")

processed_indicator_paths, skipped_indicator_paths, failed_indicator_paths
